# Elasticsearch performance: 10 checks before adding nodes

Companion notebook for [the article](https://www.elastic.co/search-labs/blog/elasticsearch-performance-before-scaling). It runs each check against a synthetic
`audit-*` dataset and deletes it at the end.

## Audit checklist

| Check | Minutes | Evidence to collect | Notebook section |
| --- | ---: | --- | ---: |
| 1. Establish the baseline | 4 | Latency, queues, processor load, heap and hot threads | 3 |
| 2. Check search scope | 2 | Target indices and shard counts | 4 |
| 3. Trim response work | 3 | Fields, total-hit requirement and response bytes | 5 |
| 4. Review pagination | 3 | Page depth and export behavior | 6 |
| 5. Review aggregation size | 3 | Top results versus complete bucket enumeration | 7 |
| 6. Inspect flattened aggregations | 3 | Mapping, metadata cardinality and aggregation latency | 8 |
| 7. Check nested semantics | 3 | Whether queries need relationships within each object | 9 |
| 8. Inspect reindex concurrency | 3 | Slices, source shards and destination settings | 10 |
| 9. Check refresh requirements | 3 | Visibility delay and forced refreshes | 11 |
| 10. Count repeated updates | 3 | Changes per ID and primary indexing operations | 12 |

## Prerequisites

- Elasticsearch 9.5 or later on Elastic Cloud Hosted or a self-managed cluster (not Serverless).
  Use a test cluster: the notebook writes about 300 MB and runs reindex benchmarks.
- A `.env` file copied from `.env.example`, with an API key created in Kibana **Dev Tools**:

```
POST /_security/api_key
{
  "name": "elasticsearch-performance-before-scaling-notebook",
  "role_descriptors": {
    "audit-lab": {
      "cluster": ["monitor"],
      "indices": [{"names": ["audit-*"], "privileges": ["all"]}]
    }
  }
}
```

Use the `encoded` value of the response as `ELASTICSEARCH_API_KEY`.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import statistics
import time
import warnings
from collections.abc import Iterable
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"].rstrip("/")
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]

es = Elasticsearch(
    ES_URL, api_key=API_KEY, request_timeout=300, retry_on_status=(429,), max_retries=5
)

OCTOBER = "audit-events-2026-10"
SEPTEMBER = "audit-events-2026-09"
OBSERVATIONS = "audit-observations"
OCTOBER_DOCS = 60_000
SEPTEMBER_DOCS = 6_000

# Everything this notebook creates. Clean up deletes exactly these names.
CREATED = [
    OBSERVATIONS,
    OCTOBER,
    SEPTEMBER,
    "audit-flattened-small",
    "audit-flattened-volume",
    "audit-flattened-cardinality",
    "audit-array-object",
    "audit-array-nested",
    *[f"audit-copy-{kind}-{n}" for kind in ("single", "auto") for n in range(3)],
    "audit-copy-demo",
    "audit-refresh",
    "audit-refresh-each",
    "audit-refresh-bulk",
    "audit-updates-each",
    "audit-updates-in-one-bulk",
    "audit-updates-coalesced",
]

EVENT_PROPERTIES = {
    "event_id": {"type": "long"},
    "timestamp": {"type": "date"},
    "device": {"type": "keyword"},
    "region": {"type": "keyword"},
    "status": {"type": "keyword"},
    "labels": {"type": "flattened"},
    "payload": {"type": "text", "index": False},
}


def create_index(
    name: str, properties: dict, shards: int = 1, refresh_interval: str = "-1"
) -> None:
    """Create an index with no replicas and an explicit refresh interval."""
    es.indices.create(
        index=name,
        settings={
            "number_of_shards": shards,
            "number_of_replicas": 0,
            "refresh_interval": refresh_interval,
        },
        mappings={"properties": properties},
    )


def bulk_docs(
    index: str,
    docs: Iterable[tuple[int, dict]],
    op: str = "index",
    refresh: str = "false",
) -> None:
    """Send (id, document) pairs with _bulk as index or partial-update actions. Raises on any failure."""
    if op not in ("index", "update"):
        raise ValueError(f"unsupported op: {op}")
    body_key = "doc" if op == "update" else "_source"
    actions = (
        {"_op_type": op, "_index": index, "_id": str(ident), body_key: doc}
        for ident, doc in docs
    )
    helpers.bulk(
        es, actions, chunk_size=1000, refresh=refresh, max_retries=5, initial_backoff=2
    )


def primaries(index: str, metric: str, key: str) -> int:
    """Read a primaries counter from the index stats API."""
    return es.indices.stats(index=index, metric=metric)["_all"]["primaries"][metric][
        key
    ]


def record(name: str, value: dict) -> None:
    """Store a measurement in audit-observations and print it."""
    document = {
        "experiment": name,
        "measured_at": datetime.now(timezone.utc).isoformat(),
        "result": value,
    }
    es.index(index=OBSERVATIONS, id=name, document=document, refresh=True)
    print(json.dumps({name: value}, indent=2))


info = es.info()
print(f"Elasticsearch {info['version']['number']}, cluster {info['cluster_name']}")

## 2. Create the lab data

Stops if any `audit-*` index already exists: run **Clean up** first to start over.

In [ ]:
existing = es.indices.get(
    index="audit-*", allow_no_indices=True, expand_wildcards="all"
)
if existing:
    raise RuntimeError(
        f"audit-* indices already exist: {sorted(existing)}. Run the Clean up cell first."
    )

create_index(
    OBSERVATIONS,
    {"result": {"type": "object", "enabled": False}},
    refresh_interval="1s",
)
create_index(OCTOBER, EVENT_PROPERTIES, shards=4)
create_index(SEPTEMBER, EVENT_PROPERTIES, shards=4)


def event(i: int, timestamp: str) -> dict:
    """Deterministic event: 1,000 devices, 6 regions and a 2 KiB payload the list never shows."""
    return {
        "event_id": i,
        "timestamp": timestamp,
        "device": f"device-{i % 1000:04}",
        "region": f"region-{(i // 1000) % 6}",
        "status": "ok" if i % 4 else "warning",
        "labels": {"latency_ms": str([2, 10, 100][i % 3]), "producer": "sensor"},
        "payload": "telemetry " + "x" * 2048,
    }


started = time.perf_counter()
bulk_docs(OCTOBER, ((i, event(i, "2026-10-01T00:00:00Z")) for i in range(OCTOBER_DOCS)))
bulk_docs(
    SEPTEMBER, ((i, event(i, "2026-09-01T00:00:00Z")) for i in range(SEPTEMBER_DOCS))
)
es.indices.refresh(index="audit-events-*")
es.indices.put_settings(
    index="audit-events-*", settings={"index.refresh_interval": "1s"}
)

assert es.count(index=OCTOBER)["count"] == OCTOBER_DOCS
assert es.count(index=SEPTEMBER)["count"] == SEPTEMBER_DOCS
print(
    f"Indexed {OCTOBER_DOCS:,} + {SEPTEMBER_DOCS:,} events in {time.perf_counter() - started:.0f} s"
)

## 3. Establish a baseline while the workload is running

In production, take this sample during the slowdown and compare counter deltas.

In [ ]:
node_stats = es.nodes.stats(
    node_id="_local",
    metric="os,jvm,thread_pool",
    filter_path=",".join(
        [
            "nodes.*.name",
            "nodes.*.os.cpu.percent",
            "nodes.*.jvm.mem.heap_used_percent",
            "nodes.*.jvm.gc.collectors.*.collection_time_in_millis",
            "nodes.*.thread_pool.search.queue",
            "nodes.*.thread_pool.search.rejected",
            "nodes.*.thread_pool.write.queue",
            "nodes.*.thread_pool.write.rejected",
        ]
    ),
)
print(json.dumps(node_stats.body, indent=2))

hot_threads = es.nodes.hot_threads(threads=3).body
print("\nHot threads (first 20 lines):")
print("\n".join(hot_threads.splitlines()[:20]))

In [ ]:
def profile_ms(shard: dict, part: str) -> float:
    """Total time of the query or aggregation tree of one profiled shard, in milliseconds."""
    if part == "query":
        return (
            sum(
                q["time_in_nanos"]
                for search in shard["searches"]
                for q in search["query"]
            )
            / 1e6
        )
    return sum(a["time_in_nanos"] for a in shard["aggregations"]) / 1e6


match_all = es.search(
    index=OCTOBER, query={"match_all": {}}, size=0, profile=True, request_cache=False
)
for shard in match_all["profile"]["shards"]:
    print(f"match_all {shard['id']}: query {profile_ms(shard, 'query'):.3f} ms")

filtered = es.search(
    index=OCTOBER,
    size=0,
    profile=True,
    request_cache=False,
    query={"term": {"status": "warning"}},
    aggs={"regions": {"terms": {"field": "region"}}},
)
for shard in filtered["profile"]["shards"]:
    print(
        f"term + terms {shard['id']}: query {profile_ms(shard, 'query'):.3f} ms, "
        f"aggregation {profile_ms(shard, 'aggregation'):.3f} ms"
    )

## 4. Search only the indices the request needs

In [ ]:
scope = es.msearch(
    searches=[
        {"index": "audit-events-*"},
        {"size": 0},
        {"index": OCTOBER},
        {"size": 0},
    ],
    filter_path="responses._shards",
)
print(json.dumps(scope.body, indent=2))

broad_shards, narrow_shards = (r["_shards"]["total"] for r in scope["responses"])
assert (broad_shards, narrow_shards) == (8, 4)
record(
    "shard-scope",
    {"broad_total_shards": broad_shards, "narrow_total_shards": narrow_shards},
)

## 5. Stop returning fields and counts the screen doesn't use

Byte counts are uncompressed JSON (`Accept-Encoding: identity`).

In [ ]:
def search_bytes(body: dict) -> tuple[dict, int]:
    """Run a search over plain HTTP and return the parsed response and its size in bytes."""
    response = requests.post(
        f"{ES_URL}/{OCTOBER}/_search",
        headers={
            "Authorization": f"ApiKey {API_KEY}",
            "Content-Type": "application/json",
            "Accept-Encoding": "identity",
        },
        json=body,
        timeout=120,
    )
    response.raise_for_status()
    return response.json(), len(response.content)


page = {"size": 100, "sort": [{"event_id": "asc"}]}
full, full_bytes = search_bytes({**page, "track_total_hits": True})
lean, lean_bytes = search_bytes(
    {**page, "track_total_hits": False, "_source": ["event_id", "device", "status"]}
)

same_hits = [h["_id"] for h in full["hits"]["hits"]] == [
    h["_id"] for h in lean["hits"]["hits"]
]
assert same_hits
record(
    "response-payload",
    {
        "page_hits": 100,
        "full_json_bytes": full_bytes,
        "filtered_json_bytes": lean_bytes,
        "same_hit_ids": same_hits,
        "http_compression": False,
        "exact_total_requested_after": False,
    },
)
print(f"Reduction: {1 - lean_bytes / full_bytes:.1%}")

## 6. Separate browsing from exporting

In [ ]:
window = es.options(ignore_status=400).search(index=OCTOBER, from_=10_000, size=100)
print(json.dumps(window["error"]["root_cause"], indent=2))
assert window.meta.status == 400

PAGE_SIZE = 1000
seen_ids, hits_read, export_pages, cursor = set(), 0, 0, None
pit_id = es.open_point_in_time(index=OCTOBER, keep_alive="2m")["id"]
try:
    while True:
        after = {"search_after": cursor} if cursor is not None else {}
        response = es.search(
            size=PAGE_SIZE,
            track_total_hits=False,
            source=False,
            pit={"id": pit_id, "keep_alive": "2m"},
            sort=["_shard_doc"],
            **after,
        )
        pit_id = response.get("pit_id", pit_id)
        hits = response["hits"]["hits"]
        if not hits:
            break
        export_pages += 1
        hits_read += len(hits)
        seen_ids.update(hit["_id"] for hit in hits)
        cursor = hits[-1]["sort"]
finally:
    es.close_point_in_time(id=pit_id)

duplicates = hits_read - len(seen_ids)
assert len(seen_ids) == OCTOBER_DOCS and duplicates == 0
record(
    "pit-pagination",
    {
        "unique_hits": len(seen_ids),
        "nonempty_pages": export_pages,
        "page_size": PAGE_SIZE,
        "duplicate_hits": duplicates,
        "pit_closed": True,
    },
)

## 7. Paginate reports that need every aggregation bucket

In [ ]:
SOURCES = [
    {"region": {"terms": {"field": "region"}}},
    {"device": {"terms": {"field": "device"}}},
]
BUCKETS_PER_PAGE = 500

pairs, bucket_docs, report_pages, after_key = set(), 0, 0, None
while True:
    composite = {"size": BUCKETS_PER_PAGE, "sources": SOURCES}
    if after_key is not None:
        composite["after"] = after_key
    response = es.search(
        index=OCTOBER,
        size=0,
        track_total_hits=False,
        request_cache=False,
        aggs={"pairs": {"composite": composite}},
    )
    group = response["aggregations"]["pairs"]
    if not group["buckets"]:
        break
    report_pages += 1
    for bucket in group["buckets"]:
        pairs.add(tuple(bucket["key"].values()))
        bucket_docs += bucket["doc_count"]
    after_key = group.get("after_key")
    if after_key is None:
        break

assert len(pairs) == 6_000 and bucket_docs == OCTOBER_DOCS
record(
    "composite-pagination",
    {
        "unique_buckets": len(pairs),
        "nonempty_pages": report_pages,
        "page_size": BUCKETS_PER_PAGE,
        "sum_doc_count": bucket_docs,
    },
)

# A two-bucket page, as in the article's screenshot.
first_page = es.search(
    index=OCTOBER,
    size=0,
    track_total_hits=False,
    filter_path="aggregations",
    aggs={"pairs": {"composite": {"size": 2, "sources": SOURCES}}},
)
print(json.dumps(first_page.body, indent=2))

## 8. Check flattened fields behind slow aggregations

Compares a `terms` aggregation on `labels.status` (flattened) and on `status` (keyword) as volume
and metadata cardinality grow. Timings depend on your cluster.

In [ ]:
STATUSES = ["closed", "open", "pending", "resolved"]
# (index suffix, documents, distinct values per extra label)
CASES = [
    ("small", 10_000, 100),
    ("volume", 100_000, 100),
    ("cardinality", 100_000, 100_000),
]
ROUNDS = 9


def labeled_doc(i: int, distinct: int) -> dict:
    """Same status in the keyword and flattened fields, plus three labels with `distinct` values."""
    status = STATUSES[i % len(STATUSES)]
    labels = {
        key: f"{key}-{i % distinct:06}"
        for key in ("trace_id", "session_id", "request_id")
    }
    return {"status": status, "labels": {"status": status, **labels}}


def terms_agg(index: str, field: str) -> dict:
    """Uncached terms aggregation over the four statuses."""
    aggs = {"statuses": {"terms": {"field": field, "size": len(STATUSES)}}}
    return es.search(
        index=index, size=0, track_total_hits=False, request_cache=False, aggs=aggs
    )


FIELDS = {"flattened": "labels.status", "keyword": "status"}
growth = {}
for suffix, count, distinct in CASES:
    index = f"audit-flattened-{suffix}"
    create_index(
        index, {"labels": {"type": "flattened"}, "status": {"type": "keyword"}}
    )
    batch = count // 4  # four refreshed batches, so each shard has several segments
    for start in range(0, count, batch):
        bulk_docs(
            index, ((i, labeled_doc(i, distinct)) for i in range(start, start + batch))
        )
        es.indices.refresh(index=index)
    assert es.count(index=index)["count"] == count

    expected = [{"key": s, "doc_count": count // len(STATUSES)} for s in STATUSES]
    for field in FIELDS.values():  # warm-up
        terms_agg(index, field)

    first_ms = {kind: [] for kind in FIELDS}
    repeat_ms = {kind: [] for kind in FIELDS}
    for round_no in range(ROUNDS):
        for kind in sorted(FIELDS, reverse=round_no % 2 == 1):  # alternate the order
            es.indices.clear_cache(index=index, fielddata=True)  # drop global ordinals
            first, repeat = terms_agg(index, FIELDS[kind]), terms_agg(
                index, FIELDS[kind]
            )
            assert first["aggregations"]["statuses"]["buckets"] == expected
            assert repeat["aggregations"]["statuses"]["buckets"] == expected
            first_ms[kind].append(first["took"])
            repeat_ms[kind].append(repeat["took"])

    medians = {f"{kind}_first_ms": statistics.median(first_ms[kind]) for kind in FIELDS}
    medians |= {
        f"{kind}_repeat_ms": statistics.median(repeat_ms[kind]) for kind in FIELDS
    }
    growth[f"{count}_docs_{3 * distinct}_other_values"] = medians

record("flattened-growth", growth)

In [ ]:
mapping = es.indices.get_mapping(
    index="audit-flattened-cardinality", filter_path="*.mappings.properties"
)
print(json.dumps(mapping.body, indent=2))

## 9. Use nested fields when the query needs object relationships

In [ ]:
ARRAY_INDICES = ["audit-array-object", "audit-array-nested"]
ARRAY_DOCS = 1000
item_properties = {"name": {"type": "keyword"}, "quantity": {"type": "integer"}}
items = {
    "items": [
        {"name": "red", "quantity": 1},
        {"name": "blue", "quantity": 5},
        {"name": "green", "quantity": 9},
    ]
}

create_index(
    "audit-array-object", {"items": {"type": "object", "properties": item_properties}}
)
create_index(
    "audit-array-nested", {"items": {"type": "nested", "properties": item_properties}}
)
for index in ARRAY_INDICES:
    bulk_docs(index, ((i, items) for i in range(ARRAY_DOCS)))
    es.indices.refresh(index=index)

physical = es.indices.stats(
    index="audit-array-*", metric="docs", filter_path="indices.*.primaries.docs.count"
)
print(json.dumps(physical.body, indent=2))

red_and_5 = {
    "bool": {
        "filter": [{"term": {"items.name": "red"}}, {"term": {"items.quantity": 5}}]
    }
}
physical_counts = {
    i: physical["indices"][i]["primaries"]["docs"]["count"] for i in ARRAY_INDICES
}
logical_counts = {i: es.count(index=i)["count"] for i in ARRAY_INDICES}
object_hits = es.count(index="audit-array-object", query=red_and_5)["count"]
nested_hits = es.count(
    index="audit-array-nested", query={"nested": {"path": "items", "query": red_and_5}}
)["count"]

assert physical_counts == {
    "audit-array-object": ARRAY_DOCS,
    "audit-array-nested": 4 * ARRAY_DOCS,
}
assert set(logical_counts.values()) == {ARRAY_DOCS}
assert (object_hits, nested_hits) == (ARRAY_DOCS, 0)
record(
    "array-semantics",
    {
        "physical_document_counts": physical_counts,
        "logical_document_counts": logical_counts,
        "red_and_quantity_5_object_matches": object_hits,
        "red_and_quantity_5_nested_matches": nested_hits,
    },
)

## 10. Check how the reindex job uses concurrency

The second cell compares an unsliced reindex with `slices=auto`. Timings depend on your cluster.

In [ ]:
create_index("audit-copy-demo", EVENT_PROPERTIES, shards=4)
task_id = es.reindex(
    source={"index": OCTOBER},
    dest={"index": "audit-copy-demo"},
    slices="auto",
    wait_for_completion=False,
)["task"]
with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore"
    )  # the client flags the tasks API as technical preview
    task = es.tasks.get(task_id=task_id, wait_for_completion=True, timeout="10m")
assert task["completed"] and not task.get("error") and not task["response"]["failures"]
print(
    f"Task {task_id}: {task['response']['created']:,} documents in {task['response']['took']} ms"
)

es.indices.refresh(index="audit-copy-demo")
assert es.count(index="audit-copy-demo")["count"] == OCTOBER_DOCS
es.indices.put_settings(
    index="audit-copy-demo", settings={"index.refresh_interval": "1s"}
)

In [ ]:
REINDEX_ROUNDS = 3
reindex_runs = []
for round_no in range(REINDEX_ROUNDS):
    for sliced in (
        [False, True] if round_no % 2 == 0 else [True, False]
    ):  # alternate the order
        destination = f"audit-copy-{'auto' if sliced else 'single'}-{round_no}"
        create_index(destination, EVENT_PROPERTIES, shards=4)
        slicing = {"slices": "auto"} if sliced else {}
        response = es.options(request_timeout=900).reindex(
            source={"index": OCTOBER},
            dest={"index": destination},
            wait_for_completion=True,
            **slicing,
        )
        assert response["created"] == OCTOBER_DOCS and not response["failures"]
        reindex_runs.append(
            {
                "round": round_no + 1,
                "slices": "auto" if sliced else 1,
                "took_ms": response["took"],
            }
        )
        es.indices.delete(index=destination)
        print(reindex_runs[-1])

record(
    "reindex-slicing",
    {
        "source_primary_shards": 4,
        "destination_primary_shards": 4,
        "documents_per_run": OCTOBER_DOCS,
        "trials": reindex_runs,
        "unsliced_median_took_ms": statistics.median(
            r["took_ms"] for r in reindex_runs if r["slices"] == 1
        ),
        "auto_median_took_ms": statistics.median(
            r["took_ms"] for r in reindex_runs if r["slices"] == "auto"
        ),
    },
)

## 11. Match refresh behavior to the visibility requirement

In [ ]:
# Try a longer interval, then restore the original setting.
settings = es.indices.get_settings(index=OCTOBER, name="index.refresh_interval")
original_interval = settings[OCTOBER]["settings"]["index"]["refresh_interval"]
es.indices.put_settings(index=OCTOBER, settings={"index.refresh_interval": "30s"})
print(es.indices.get_settings(index=OCTOBER, name="index.refresh_interval").body)
es.indices.put_settings(
    index=OCTOBER, settings={"index.refresh_interval": original_interval}
)

# An acknowledged document is visible to a real-time GET before a refresh makes it searchable.
create_index("audit-refresh", {"value": {"type": "integer"}})
es.index(index="audit-refresh", id="one", document={"value": 1})
realtime_found = es.get(index="audit-refresh", id="one")["found"]
count_before = es.count(index="audit-refresh")["count"]
es.indices.refresh(index="audit-refresh")
count_after = es.count(index="audit-refresh")["count"]
assert realtime_found and (count_before, count_after) == (0, 1)

# 100 writes with refresh=true versus one bulk of 100 with refresh=true (scheduled refresh disabled).
WRITES = 100
refresh_indices = ["audit-refresh-each", "audit-refresh-bulk"]
for index in refresh_indices:
    create_index(index, {"value": {"type": "integer"}})
refresh_before = {i: primaries(i, "refresh", "total") for i in refresh_indices}
for i in range(WRITES):
    es.index(index="audit-refresh-each", id=str(i), document={"value": i}, refresh=True)
bulk_docs(
    "audit-refresh-bulk", ((i, {"value": i}) for i in range(WRITES)), refresh="true"
)
refresh_deltas = {
    i: primaries(i, "refresh", "total") - before for i, before in refresh_before.items()
}

record(
    "refresh-visibility",
    {
        "realtime_get_found_before_refresh": realtime_found,
        "search_count_before_refresh": count_before,
        "search_count_after_refresh": count_after,
        "writes_each_case": WRITES,
        "refresh_total_deltas": refresh_deltas,
        "auto_refresh_disabled_for_both_comparison_indices": True,
    },
)

## 12. Combine repeated changes to the same document

In [ ]:
ENTITIES, CHANGES = 100, 10
update_indices = [
    "audit-updates-each",
    "audit-updates-in-one-bulk",
    "audit-updates-coalesced",
]
for index in update_indices:
    create_index(index, {"temperature": {"type": "integer"}})
    bulk_docs(index, ((i, {"temperature": 0}) for i in range(ENTITIES)))
indexing_before = {i: primaries(i, "indexing", "index_total") for i in update_indices}

# One bulk per change, all changes in one bulk, and one coalesced update per document.
for value in range(1, CHANGES + 1):
    bulk_docs(
        "audit-updates-each",
        ((i, {"temperature": value}) for i in range(ENTITIES)),
        op="update",
    )
bulk_docs(
    "audit-updates-in-one-bulk",
    (
        (i, {"temperature": value})
        for value in range(1, CHANGES + 1)
        for i in range(ENTITIES)
    ),
    op="update",
)
bulk_docs(
    "audit-updates-coalesced",
    ((i, {"temperature": CHANGES}) for i in range(ENTITIES)),
    op="update",
)

update_deltas = {
    i: primaries(i, "indexing", "index_total") - before
    for i, before in indexing_before.items()
}
for index in update_indices:
    docs = es.mget(index=index, ids=[str(i) for i in range(ENTITIES)])["docs"]
    assert all(d["_source"]["temperature"] == CHANGES for d in docs), index
assert update_deltas == {
    "audit-updates-each": ENTITIES * CHANGES,
    "audit-updates-in-one-bulk": ENTITIES * CHANGES,
    "audit-updates-coalesced": ENTITIES,
}

# The value is already 10, so the update is a no-op.
noop = es.update(index="audit-updates-coalesced", id="0", doc={"temperature": CHANGES})
assert noop["result"] == "noop"
record(
    "update-coalescing",
    {
        "entities": ENTITIES,
        "changes_per_entity": CHANGES,
        "primary_indexing_operation_deltas": update_deltas,
        "final_temperature_both_cases": CHANGES,
        "repeated_identical_doc_result": noop["result"],
    },
)

## 13. Review the recorded measurements

In [ ]:
observations = es.search(index=OBSERVATIONS, size=20)["hits"]["hits"]
for hit in sorted(observations, key=lambda h: h["_id"]):
    print(f"{hit['_id']}: {json.dumps(hit['_source']['result'])}")

## 14. Clean up

In [ ]:
es.indices.delete(index=",".join(CREATED), ignore_unavailable=True)
left = es.indices.get(index="audit-*", allow_no_indices=True, expand_wildcards="all")
print(
    "Done"
    if not left
    else f"Still present (not created by this notebook): {sorted(left)}"
)